# 构建Vul知识库所需的数据集

In [ ]:
import os
import sys
import yaml
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sklearn.cluster import KMeans
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
print("HF_ENDPOINT =", os.environ.get("HF_ENDPOINT"))

# 向上两级到项目根目录
PROJECT_ROOT = Path(os.path.abspath("")).resolve()
while not (PROJECT_ROOT / "configs").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.retriever.Base import create_embedder

In [ ]:
# ===========================================================
# Load the Jina code embedding model from config
# ===========================================================

# with open(f"{PROJECT_ROOT}/configs/models/jina-code-embeddings-0.5b.yml", "r") as f:
#     model_config = yaml.safe_load(f)
with open(f"{PROJECT_ROOT}/configs/models/harrier-oss-v1-0.6b.yml", "r") as f:
    model_config = yaml.safe_load(f)

embedder = create_embedder(model_config)
device = model_config.get("device", "cuda")

# ===========================================================
# Load the dataset
# ===========================================================

# 加载并处理BigVul数据集
bigvul_df = pd.read_json(f"{PROJECT_ROOT}/data/raw/vul/BigVul.jsonl", lines=True)
primevul_df = pd.read_json(f"{PROJECT_ROOT}/data/raw/vul/PrimeVul.jsonl", lines=True)


# 合并数据集
df = pd.concat([bigvul_df, primevul_df], ignore_index=True)
df

In [ ]:
# 按照vul_code对df内的数据进行去重
before = len(df)
df = df.drop_duplicates(subset=['vul_code'], keep='first').reset_index(drop=True)
after = len(df)
print(f"按 vul_code 去重: {before} -> {after} (移除 {before - after})")
df

In [ ]:
# 将vul_code进行格式化（多进程并行加速）
from src.utils.formatter import auto_format_c_code_batch

codes = df['vul_code'].astype(str).tolist()
results = auto_format_c_code_batch(codes, on_error="keep")
df['vul_code'] = [formatted for formatted, _ in results]
print(f"格式化完成，共处理 {len(codes)} 条记录")
df.head()

In [ ]:
# 一次性传入全量代码列表，由嵌入模型内部按 batch_size 分批编码
all_codes = df['vul_code'].astype(str).tolist()

encoded_code_vectors = embedder.embed_documents(all_codes)
encoded_code_vectors = np.asarray(encoded_code_vectors)

# 建立 idx -> 向量的映射（与 df 行顺序一致）
index_to_vector_map = {int(idx_val): encoded_code_vectors[i] for i, idx_val in enumerate(df['id'])}
print("encoded_code_vectors shape:", encoded_code_vectors.shape)

# 保留 tensor 形式
encoded_code_vectors = torch.tensor(encoded_code_vectors)

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42)
print("Fitting KMeans on embeddings with shape:", encoded_code_vectors.shape)
kmeans.fit(encoded_code_vectors)

# Get the cluster labels for each code piece
cluster_labels = kmeans.labels_
print("KMeans finished. cluster_labels shape:", cluster_labels.shape)

In [ ]:
# 写入聚类标签（改名为 cluster_id）
df['cluster_id'] = cluster_labels

# 所有数据写入同一个文件（不再按簇拆分）
# output_file = f'{PROJECT_ROOT}/data/corpus/vul/jina_clustered_all.jsonl'
corpus_output_file = f'{PROJECT_ROOT}/data/corpus/vul/clustered_rag_data.jsonl'
df.to_json(corpus_output_file, orient='records', lines=True)

print(f"Saved {len(df)} samples -> {corpus_output_file}")

df.head()
